# 🏗️ Notebook 1: Code Deployment (CI/CD) — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/code-deployment
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A system that turns commits into running code in production. Stages: build → test → artifact → deploy → verify. If verify fails, we roll back.

Goal: **safe, frequent deploys** with a clear audit trail.

## Requirements

### Functional
- Trigger on push/PR/manual.
- Run a DAG of stages in parallel where possible.
- Produce an **immutable artifact** (container image + SHA).
- Deploy to env (dev, staging, prod).
- Canary → ramp → full; automatic rollback on SLO breach.

### Non-functional
- Reproducible builds.
- End-to-end time < 20 min.
- No deploy silently proceeds on a red test.

## Back-of-envelope

- 10k engineers × 5 builds/day = 50k builds/day.
- Each build runs on an ephemeral worker (6 CPU / 8 GB / 15 min avg).
- Peak parallelism: ~5k workers at 10am.

## High-level architecture

```
  [Git]──webhook──► Trigger Svc ──► Pipeline Svc ──► Scheduler ──► Worker pool
                                           │
                                           ▼
                                     Artifact Registry
                                           │
                                           ▼
                                     Deployer ──► K8s / target envs
                                           │
                                           ▼
                                  Monitor (SLO/metrics)
                                           │  on breach
                                           ▼
                                     Auto-rollback
```

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.